In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from utils.ica_pipeline import *
from utils.epoch_and_average import *

from IPython.display import clear_output
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
DATA_DIR = "/Users/jowanglin/regression-based_ERP/data/eeg/crystal"
STIMULI_EXCEL = "/Users/jowanglin/regression-based_ERP/data/stimuli/CRYSTAL_master-sheet.xlsx"

NOISE_COV = None                      # Noise covariance used for pre-whitening. If None (default), channels are scaled to unit variance (“z-standardized”) as a group by channel type prior to the whitening by PCA.
RANDOM_STATE = 42                     # Fix this throughout the script -- always!!

METHOD = "infomax"                    # MNE accepts 'fastica' | 'infomax' | 'picard'; using "picard" to match EEGLAB default -> need to pip install python-picard
FIT_PARAMS={"extended": True,         # EEGLAB default is infomax extended
            "weights": None,          # The initialized unmixing matrix. Defaults to None, which means the identity matrix is used.
            "l_rate": None,           # This quantity indicates the relative size of the change in weights. Defaults to 0.01 / log(n_features ** 2).
            "block": None,            # The block size of randomly chosen data segments. Defaults to floor(sqrt(n_times / 3.)).           
            "w_change": 1e-12,        # The change at which to stop iteration. Defaults to 1e-12.
            "anneal_deg": 60.0,       # The angle (in degrees) at which the learning rate will be reduced. Defaults to 60.0.
            "anneal_step": 0.9,       # The factor by which the learning rate will be reduced. Defaults to 0.9.
            "n_subgauss": 1,          # The number of subgaussian components. Only considered for extended Infomax. Defaults to 1.
            "kurt_size": 6000,        # The window size for kurtosis estimation. Only considered for extended Infomax. Defaults to 6000.
            "blowup": 10000}          # The maximum difference allowed between two successive estimations of the unmixing matrix. Defaults to 10000.

# for picard
#FIT_PARAMS={"tol": 1e-7,
            #"ortho": False, # If True, uses Picard-O. Otherwise, uses the standard Picard.
            #"fastica_it": None} # If an int, perform fastica_it iterations of FastICA before running Picard. It might help starting from a better point.


MAX_ITER=1000 
#allow_ref_meg -> irrelevant

CORR_THRESHOLD = 0.85
VERBOSE=True 

In [16]:
write_to_path = f"{DATA_DIR}/after_ica/subj_eog_indices.txt"
f = open(write_to_path, "w")
f.write("")
f.close()

raw_clean_list, eog_indices_list = [], []
f = open(write_to_path, "a")
for num in range(1, 28):
    raw_file_name = f"subj{str(num).zfill(3)}_reref_filt.set"
    try:
        raw = mne.io.read_raw_eeglab(f"{DATA_DIR}/{raw_file_name}", preload=True, verbose=False)
    except FileNotFoundError:
        continue
    
    print(f"subj{str(num).zfill(3)}")
    raw_reref, raw_for_ica, rank = preprocess_for_ica(raw,
                                                  standard_montage="standard_1020",
                                                  eog_channels=["HEO", "VEO"],
                                                  ref_channels=["M1", "M2"],
                                                  time_threshold=3.0,
                                                  after_event_code_buffer=1.5,
                                                  before_event_code_buffer=0.5,
                                                  rank_tol=1e-4,
                                                  verbose=False)
    raw_clean, eog_indices, _ = perform_ica(raw_to_clean=raw_reref,
                                            raw_for_ica=raw_for_ica,
                                            n_components=rank,
                                            noise_cov=NOISE_COV,
                                            method=METHOD,
                                            fit_params=FIT_PARAMS,
                                            max_iter=MAX_ITER,
                                            manual_inspection=False,
                                            corr_threshold=CORR_THRESHOLD,
                                            eog_like_channels=["VEO"], 
                                            verbose=False)
    raw_clean.save(f"{DATA_DIR}/after_ica/subj{str(num).zfill(3)}_after_ica_raw.fif", overwrite=True)
    raw_clean_list.append(raw_clean)
    
    f.write(f"subj{str(num).zfill(3)}: " + str(eog_indices) + "\n")
    eog_indices_list.append(eog_indices)
f.close()

clear_output()


In [17]:
# sanity checking channel positions are set, channel names are upper case, and that annotations are not lost
raw_check = raw_clean_list[0]

m = raw_check.get_montage()
pos = m.get_positions()["ch_pos"]
print(pos["FP1"])
display(pd.DataFrame(raw_check.annotations).head(10))

[-0.0309026   0.11458518  0.02786657]


,onset,duration,description,orig_time,extras
0,12.276,0.0,1,None,{}
1,25.898,0.0,210,None,{}
2,26.964,0.0,79,None,{}
3,27.324,0.0,220,None,{}
4,27.684,0.0,220,None,{}
5,28.044,0.0,220,None,{}
6,28.404,0.0,220,None,{}
7,28.764,0.0,220,None,{}
8,29.124,0.0,220,None,{}
9,29.484,0.0,79,None,{}


In [20]:
print(len(eog_indices_list))
print(eog_indices_list)

20
[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []]


In [7]:
df_stimuli = pd.read_excel(STIMULI_EXCEL, sheet_name="Overall")
fixation = "210"
non_final = "220"
item_codes = df_stimuli["W1_code"].unique().astype(str)
high_constraint = ("240", "241", "242", "243")  # just testing that both str and int work
low_constraint = ("244", "245", "246", "247")
emo_word = (240, 241, 244, 245)
neu_word = (242, 243, 246, 247)
conditions_list = [{"high_constraint": high_constraint, "low_constraint": low_constraint},
                   {"emo_word": emo_word, "neu_word": neu_word}]

subj_epochs_list = []
for num in range(1, 28):
    raw_file_name = f"subj{str(num).zfill(3)}_after_ica_raw.fif"
    try:
        raw = mne.io.read_raw_fif(f"{DATA_DIR}/after_ica/{raw_file_name}", preload=True, verbose=False)
    except FileNotFoundError:
        continue

    print(f"subj{str(num).zfill(3)}")
    epochs_list = eeglab_logic_bin_epoch(raw.copy(),
                                         fixation=fixation,
                                         non_final=non_final,
                                         item_codes=item_codes,
                                         position_range=(1, 7),  # words 1 to 7
                                         conditions_list=conditions_list,
                                         tmin=-0.1, tmax=0.7,
                                         baseline=(-0.1, 0),
                                         reject_peak_to_peak={"eeg": 200e-6,
                                                              "eog": 200e-6},
                                         resample=None,
                                         preload=True,
                                         verbose=True)
    subj_epochs_list.append(epochs_list)
clear_output()


In [10]:
subj_drop_logs = [list(filter(lambda t: False if "IGNORED" in t or len(t) == 0 else True,
                              e[0].drop_log))
                              for e in subj_epochs_list]
how_many_dropped = [len(drop_log) for drop_log in subj_drop_logs]
print(how_many_dropped)

[1, 31, 24, 14, 15, 0, 2, 2, 0, 5, 0, 6, 0, 3, 0, 4, 1, 12, 3, 15]
